# UK Crime Analysis – Data Preprocessing

This notebook performs the data extraction and preprocessing for the UK crime analysis project.

The objective of this stage is to prepare the crime dataset obtained from data.police.uk for exploratory data analysis (EDA).

The preprocessing process includes:

- Inspecting the structure of the dataset
- Assessing data quality
- Identifying relevant fields for analysis
- Preparing the dataset for further analysis

## Initial Dataset Exploration

Before processing the full dataset, a single sample dataset is explored to understand the structure and contents of the crime data.

Working with a small sample dataset first allows the structure of the data to be inspected before scaling the workflow to multiple datasets across different police forces and time periods.

This step helps to:

- Understand the dataset structure
- Identify available variables
- Inspect data types
- Assess data quality

### Import Required Libraries

The pandas library is used for data manipulation and analysis.

In [ ]:
import pandas as pd

### Define File Paths

Relative file paths are used to ensure that the notebook can be run on other machines without modification.

In [ ]:
DATA_SAMPLE = "../data_sample/"
DATA_RAW = "../data_raw/"
DATA_CLEAN = "../data_clean/"

### Load Sample Dataset

A sample dataset from January 2026 for the Metropolitan Police is loaded. Each row in the dataset represents a single recorded crime incident.

In [ ]:
df_sample = pd.read_csv(DATA_SAMPLE + "2026-01-metropolitan-street.csv")

### Preview the Dataset

The first few rows of the dataset are displayed to understand the structure of the data and how individual crime incidents are recorded.

In [ ]:
df_sample.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,7ac95c1005827988b6891417ae9263265ba6bddc0fb97e...,2026-01,Metropolitan Police Service,Metropolitan Police Service,-1.488662,52.980060,On or near Eaton Court,E01019425,Amber Valley 016A,Other theft,Under investigation,NaN
1,765ec4496c08741588a66093201a789dcf2cc932e196e4...,2026-01,Metropolitan Police Service,Metropolitan Police Service,-1.227044,53.034804,On or near Springfield Road,E01027944,Ashfield 015E,Violence and sexual offences,Under investigation,NaN
2,d44456519af612e1ddd8711f4ad1aa3f298ae81876764a...,2026-01,Metropolitan Police Service,Metropolitan Police Service,0.896070,51.161238,On or near Atkinson Walk,E01032811,Ashford 003D,Violence and sexual offences,Under investigation,NaN
3,f13ba3ef01c89c35c78e8d8a14cbd5731134c76fcd1a07...,2026-01,Metropolitan Police Service,Metropolitan Police Service,0.701809,51.133124,On or near Bethersden Road,E01024036,Ashford 011D,Violence and sexual offences,Under investigation,NaN
4,3eaacb08482f584d1cd5734f6d1d198841ab92bd5536eb...,2026-01,Metropolitan Police Service,Metropolitan Police Service,0.751960,52.035235,On or near Chelsworth Avenue,E01029888,Babergh 008C,Violence and sexual offences,Under investigation,NaN


### Dataset Dimensions

The shape of the dataset is inspected to determine the number of observations (rows) and variables (columns).

In [ ]:
df_sample.shape

(86692, 12)

### Dataset Structure

The dataset structure is examined to identify the available columns, their data types, and the presence of any missing values.

In [ ]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86692 entries, 0 to 86691
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Crime ID               70408 non-null  object 
 1   Month                  86692 non-null  object 
 2   Reported by            86692 non-null  object 
 3   Falls within           86692 non-null  object 
 4   Longitude              86692 non-null  float64
 5   Latitude               86692 non-null  float64
 6   Location               86692 non-null  object 
 7   LSOA code              86692 non-null  object 
 8   LSOA name              86692 non-null  object 
 9   Crime type             86692 non-null  object 
 10  Last outcome category  70408 non-null  object 
 11  Context                0 non-null      float64
dtypes: float64(3), object(9)
memory usage: 7.9+ MB


### Check Missing Values

Checking for missing values is important for identifying potential data quality issues. Particular attention is given to location fields such as latitude and longitude, which are required for spatial analysis.

In [ ]:
df_sample.isnull().sum()

Crime ID                 16284
Month                        0
Reported by                  0
Falls within                 0
Longitude                    0
Latitude                     0
Location                     0
LSOA code                    0
LSOA name                    0
Crime type                   0
Last outcome category    16284
Context                  86692
dtype: int64

### Investigating Missing Values

A large number of rows contain missing values for both the `Crime ID` and `Last outcome category` fields.

To determine whether these missing values occur in the same rows, a comparison was performed between the two columns. This helps assess whether the missing values follow a consistent pattern within the dataset.


The result returned `True`, indicating that the missing values in both columns are perfectly aligned. This means that every row with a missing `Crime ID` also has a missing `Last outcome category`.

This pattern is consistent with the way crime data is published on data.police.uk, where some incidents may not be assigned a unique crime identifier and may not yet have an outcome recorded at the time of publication.

In [ ]:
crime_id_null = df_sample["Crime ID"].isnull()
outcome_null = df_sample["Last outcome category"].isnull()

(crime_id_null == outcome_null).all()

np.True_

### Removing Unused Columns

The `Context` column contains only missing values across all rows in the sample dataset. Since it does not provide any useful information for analysis, it is considered redundant and can be removed during preprocessing.

Before doing so, the column will be checked again after loading the full dataset to confirm that it remains entirely empty across all records.

### Exploring Crime Categories

This is done to verify that `Crime type` contains meaningul category labels with no duplicate types.

In [ ]:
df_sample["Crime type"].value_counts()

Crime type
Violence and sexual offences    22011
Anti-social behaviour           16284
Other theft                      7505
Vehicle crime                    6621
Shoplifting                      6457
Theft from the person            5825
Drugs                            4431
Burglary                         4255
Criminal damage and arson        4230
Public order                     4229
Robbery                          2425
Other crime                      1050
Bicycle theft                     844
Possession of weapons             525
Name: count, dtype: int64

### Summary

The initial exploration of the sample dataset provided an overview of the dataset structure, available variables and potential data quality considerations.

The dataset contains one row per recorded crime incident, with variables describing the crime category, location and outcome.

Having established the structure of the data, the next step is to load and combine multiple datasets covering different police forces over a 24 month time period.

## Loading the Full Dataset

After exploring the structure of a sample dataset, the next step is to load the full set of crime datasets covering multiple police forces and time periods.

The datasets are stored as individual CSV files in the `data_raw` directory. Each file represents one month of street-level crime data for a specific police force.

### Import Additional Libraries

Additional libraries are required to automatically locate and load multiple CSV files from the raw data directory.

In [ ]:
import glob

### Locate Dataset Files

All CSV files within the `data_raw` directory are identified using a file pattern search. This ensures that all available datasets are included in the loading process.

In [ ]:
csv_files = glob.glob("../data_raw/*.csv")
len(csv_files)

96

### Load Dataset Files

Each CSV file is loaded into a Pandas DataFrame. The datasets are stored in a list so they can later be combined into a single dataset.

In [ ]:
dataframes = []

for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)

### Verify Loaded Datasets

A quick check is performed to confirm that all datasets have been loaded successfully.

In [ ]:
len(dataframes)

96

### Inspect Example Dataset

One of the loaded datasets is briefly inspected to confirm that the structure matches the sample dataset previously explored.

In [ ]:
dataframes[0].head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,ab759775ba360148fe6aa9770fe8c0a439cc7ff9a4fdf7...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.538043,54.117850,On or near Millers Ford,E01027558,Craven 001A,Criminal damage and arson,Local resolution,NaN
1,f01cb344a905e7fc8abe3f82a806e239cff1cfe4122c4b...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.545291,54.115229,On or near Eskew Crescent,E01027558,Craven 001A,Violence and sexual offences,Status update unavailable,NaN
2,NaN,2025-02,North Yorkshire Police,North Yorkshire Police,-2.509503,54.118177,On or near Supermarket,E01027559,Craven 001B,Anti-social behaviour,NaN,NaN
3,NaN,2025-02,North Yorkshire Police,North Yorkshire Police,-2.506835,54.117923,On or near Mount Pleasant,E01027559,Craven 001B,Anti-social behaviour,NaN,NaN
4,9845af834c3b6deef389879dead172dffc5e7a3a2ac286...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.509503,54.118177,On or near Supermarket,E01027559,Craven 001B,Other theft,Status update unavailable,NaN


### Verify Column Consistency

Before combining the datasets, the column structure of the individual datasets is checked to confirm that they share the same schema. This is important because combining datasets assumes that the variables are consistent across files.

Each file will compare its column names to those of the first file, the output will show if they match or not for each file.

In [ ]:
first_columns = dataframes[0].columns

for df in dataframes:
    if df.columns.equals(first_columns):
        print("No column mismatch")
    else:
        print("Column mismatch found")

No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mismatch
No column mi

### Combine Datasets

After confirming that all datasets share the same column structure, they can be combined into a single DataFrame.

This is done using vertical concatenation, where each dataset is stacked on top of the others. The resulting dataset contains all crime records across the selected police forces and time period.

In [ ]:
df_combined = pd.concat(dataframes, ignore_index=True)

In [ ]:
df_combined.shape

(3143284, 12)

In [ ]:
df_combined["Falls within"].value_counts()

Falls within
Metropolitan Police Service    2279745
West Midlands Police            664403
North Yorkshire Police          123397
Dyfed-Powys Police               75739
Name: count, dtype: int64

## Data Cleaning

After combining all datasets into a single DataFrame, the next step is to clean and prepare the data for analysis.

This includes checking for missing values, removing unnecessary columns, converting variables to appropriate data types, and creating useful time-based variables.

### Check Missing Values

Missing values are inspected to understand the completeness of the dataset and identify any variables that may require cleaning or removal.

In [ ]:
df_combined.isnull().sum()

Crime ID                  556280
Month                          0
Reported by                    0
Falls within                   0
Longitude                   4688
Latitude                    4688
Location                       0
LSOA code                   4690
LSOA name                   4690
Crime type                     0
Last outcome category     556280
Context                  3143284
dtype: int64

In [ ]:
df_combined = df_combined.drop(columns=["Context"])

In [ ]:
df_combined[df_combined["Latitude"].isnull()]["Falls within"].value_counts()

Falls within
Dyfed-Powys Police             3249
North Yorkshire Police          767
Metropolitan Police Service     672
Name: count, dtype: int64

### Handling Missing Values

The dataset contains a small number of missing values across several columns.

The `Context` column contains missing values for all records and therefore does not provide any useful information. Therefore, this column has been removed from the dataset.

The `Crime ID` and `Last outcome category` variables contain missing values because not all crimes have a recorded outcome or public identifier. These records have been retained to avoid removing valid crime events.

A very small number of records are missing geographic information (`Latitude`, `Longitude`, `LSOA code`, and `LSOA name`).

Inspection of the dataset indicates that a large proportion of these missing values occur within the Dfyed-Powys Police data. This is likely due to anonymisation practices used in rural areas where revealing approximate crime locations may risk identifying individuals or sensitive locations.

As these records represent a very small proportion of the dataset, they have been retained and will simply be excluded from geographic visualisations where location data is required.

### Convert Month to Datetime

The `Month` column originally contains year–month values stored as text. This column is converted to a datetime format stored as `Date` to enable time-based analysis.

Since the original data only specifies the year and month, pandas assigns the first day of the month when converting to a full date format (e.g., `2024-02` becomes `2024-02-01`).

The year and month can then be extracted into seperate columns

In [ ]:
df_combined["Date"] = pd.to_datetime(df_combined["Month"])

In [ ]:
df_combined["Year_Clean"] = df_combined["Date"].dt.year
df_combined["Month_Clean"] = df_combined["Date"].dt.month

In [ ]:
df_combined.head()

,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Date,Year_Clean,Month_Clean
0,ab759775ba360148fe6aa9770fe8c0a439cc7ff9a4fdf7...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.538043,54.117850,On or near Millers Ford,E01027558,Craven 001A,Criminal damage and arson,Local resolution,2025-02-01,2025,2
1,f01cb344a905e7fc8abe3f82a806e239cff1cfe4122c4b...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.545291,54.115229,On or near Eskew Crescent,E01027558,Craven 001A,Violence and sexual offences,Status update unavailable,2025-02-01,2025,2
2,NaN,2025-02,North Yorkshire Police,North Yorkshire Police,-2.509503,54.118177,On or near Supermarket,E01027559,Craven 001B,Anti-social behaviour,NaN,2025-02-01,2025,2
3,NaN,2025-02,North Yorkshire Police,North Yorkshire Police,-2.506835,54.117923,On or near Mount Pleasant,E01027559,Craven 001B,Anti-social behaviour,NaN,2025-02-01,2025,2
4,9845af834c3b6deef389879dead172dffc5e7a3a2ac286...,2025-02,North Yorkshire Police,North Yorkshire Police,-2.509503,54.118177,On or near Supermarket,E01027559,Craven 001B,Other theft,Status update unavailable,2025-02-01,2025,2


### Verify Crime Category Consistency

Before performing analysis across multiple police forces, it is important to confirm that crime categories are recorded consistently across datasets.

This check ensures that the `Crime type` variable uses the same category labels for each police force. Consistent category definitions are necessary to allow meaningful comparisons between regions in later analysis.

The results show that each police force contains the same set of crime categories, indicating that the data is consistent and suitable for comparison between regions.

In [ ]:
df_combined["Crime type"].unique()

array(['Criminal damage and arson', 'Violence and sexual offences',
       'Anti-social behaviour', 'Other theft', 'Other crime', 'Burglary',
       'Possession of weapons', 'Vehicle crime', 'Public order',
       'Shoplifting', 'Bicycle theft', 'Drugs', 'Robbery',
       'Theft from the person'], dtype=object)

In [ ]:
df_combined.groupby("Falls within")["Crime type"].nunique()

Falls within
Dyfed-Powys Police             14
Metropolitan Police Service    14
North Yorkshire Police         14
West Midlands Police           14
Name: Crime type, dtype: int64

### Sort Dataset by Date and Police Force

To improve readability and organisation of the dataset, the records are sorted first by the `Month` variable and then by the police force (`Falls within`). 

This ensures that crime records appear in chronological order while grouping the police forces consistently within each time period.

The index was also reassigned so numbers run sequentially.

In [ ]:
df_combined = df_combined.sort_values(["Date", "Falls within"])
df_combined = df_combined.reset_index(drop=True)

### Checking all months are present in the combined dataset

In [ ]:
df_combined["Date"].value_counts().sort_index()

Date
2024-01-01    124742
2024-02-01    121783
2024-03-01    125403
2024-04-01    124175
2024-05-01    137716
2024-06-01    135732
2024-07-01    141670
2024-08-01    136422
2024-09-01    131251
2024-10-01    139185
2024-11-01    132914
2024-12-01    124595
2025-01-01    120490
2025-02-01    115911
2025-03-01    130320
2025-04-01    129674
2025-05-01    137164
2025-06-01    139110
2025-07-01    146158
2025-08-01    135790
2025-09-01    127714
2025-10-01    134149
2025-11-01    128271
2025-12-01    122945
Name: count, dtype: int64

### Checking all police forces (and only those) are present in the combined dataset

In [ ]:
df_combined["Falls within"].value_counts().sort_index()

Falls within
Dyfed-Powys Police               75739
Metropolitan Police Service    2279745
North Yorkshire Police          123397
West Midlands Police            664403
Name: count, dtype: int64

### Saving cleaned dataframe to CSV

In [ ]:
df_combined.to_csv(DATA_CLEAN + "crime_cleaned.csv", index=False)

print("Cleaned dataset saved to:", DATA_CLEAN + "crime_cleaned.csv")

Cleaned dataset saved to: ../data_clean/crime_cleaned.csv
